In [ ]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from rag_helper import RAGBase

Reading documents

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1

In [ ]:
len(documents)

Q2

In [ ]:
from minsearch import Index

In [ ]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [ ]:
index = build_index(documents)

In [ ]:
index.search("How does the agentic loop keep calling the model until it stops?")[0]['filename']

Q3

In [ ]:
rag_base = RAGBase(index=index, llm_client=openai_client)

In [ ]:
response,answer = rag_base.rag("How does the agentic loop keep calling the model until it stops?")

In [ ]:
response.usage.input_tokens

Q4

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
print(len(chunks))

Q5

In [ ]:
chunks_index = build_index(chunks)

In [ ]:
chunks_rag_base = RAGBase(index=chunks_index, llm_client=openai_client)

In [ ]:
chunks_response,chunks_answer = chunks_rag_base.rag("How does the agentic loop keep calling the model until it stops?")

In [ ]:
response.usage.input_tokens // chunks_response.usage.input_tokens

Q6

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
agent_tools = Tools()

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict = {'content': 3.0, 'filename': 0.5}
    )

In [ ]:
agent_tools.add_tool(search)

In [ ]:
AGENT_INSTRUCTIONS = """You're a course teaching assistant. Answer the student's question using the search tool.
 Make multiple searches with different keywords before answering."""

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)